# Projeto Avaliativo — Machine Learning e Visão Computacional
## E-commerce Churn: previsão de abandono de clientes

Este projeto implementa um **Pipeline Preditivo completo** para identificar clientes com maior probabilidade de abandonar uma plataforma de e-commerce (`Churn = 1`), seguindo as seis fases técnicas exigidas na Situação de Aprendizagem.

**Base utilizada:** `E Commerce Dataset - E Comm.csv`

### Objetivos
- realizar Análise Exploratória de Dados (EDA);
- tratar duplicidades, valores nulos e outliers;
- criar a variável calculada `cashback_por_pedido`;
- separar treino e teste de forma estratificada;
- balancear **somente o conjunto de treino**;
- aplicar escalonamento somente no KNN;
- testar diferentes valores de `n_neighbors` e `max_depth`;
- diagnosticar overfitting comparando treino e teste;
- comparar KNN e Árvore de Decisão pelas métricas e matrizes de confusão;
- transformar os resultados em um **veredito de negócio**;
- gerar automaticamente um `README.md` completo ao final.

> **Regra metodológica:** nenhuma informação do conjunto de teste será usada para ajustar imputadores, limites de outliers, encoder, scaler ou SMOTE.

> **Versão histórica — Fase 6 — Avaliação.**
> Esta versão contém somente o que foi desenvolvido até esta fase.


## 0. Configuração do ambiente

Esta célula instala a biblioteca necessária para balanceamento de classes.

In [ ]:
!pip -q install imbalanced-learn

## 1. Importação das bibliotecas

Serão utilizadas bibliotecas de análise de dados, visualização, pré-processamento, balanceamento e modelagem.

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from google.colab import files

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix
)

from imblearn.over_sampling import SMOTE

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:.4f}")
RANDOM_STATE = 42

## 2. Carregamento do dataset

No Google Colab, o usuário fará o upload do CSV. O código identifica automaticamente o primeiro arquivo `.csv` enviado.

In [ ]:
uploaded = files.upload()

csv_files = [name for name in uploaded.keys() if name.lower().endswith(".csv")]
if not csv_files:
    raise FileNotFoundError("Nenhum arquivo CSV foi enviado.")

CSV_NAME = csv_files[0]
df_raw = pd.read_csv(CSV_NAME)

print(f"Arquivo carregado: {CSV_NAME}")
print(f"Dimensões iniciais: {df_raw.shape[0]} linhas x {df_raw.shape[1]} colunas")
display(df_raw.head())

# Fase 1 — Análise Exploratória de Dados (EDA)

A EDA documenta dimensões, tipos, estatísticas, distribuição do alvo, distribuições numéricas e correlações. As observações obtidas nesta etapa orientarão as decisões de preparação.

In [ ]:
df = df_raw.copy()

print("Dimensões:", df.shape)
print("\nTipos de dados:")
display(df.dtypes.to_frame("dtype"))

print("\nValores nulos por coluna:")
display(df.isna().sum().to_frame("nulos").sort_values("nulos", ascending=False))

print("\nResumo estatístico:")
display(df.describe(include="all").T)

### EDA — Distribuição da variável alvo

O gráfico verifica visualmente o desbalanceamento entre clientes que permaneceram e clientes que abandonaram a plataforma.

In [ ]:
target_counts = df["Churn"].value_counts().sort_index()
target_pct = df["Churn"].value_counts(normalize=True).sort_index() * 100

plt.figure(figsize=(7, 4))
bars = plt.bar(["Não Churn (0)", "Churn (1)"], target_counts.values)
plt.title("Distribuição da variável alvo — Churn")
plt.ylabel("Quantidade de clientes")
plt.xlabel("Classe")

for bar, value, pct in zip(bars, target_counts.values, target_pct.values):
    plt.text(
        bar.get_x() + bar.get_width()/2, value,
        f"{value}\n({pct:.1f}%)",
        ha="center", va="bottom"
    )

plt.tight_layout()
plt.show()

### EDA — Distribuições numéricas

Os histogramas permitem observar assimetria, concentração e possíveis valores extremos nas principais variáveis quantitativas.

In [ ]:
numeric_eda = [
    "Tenure", "WarehouseToHome", "HourSpendOnApp",
    "OrderAmountHikeFromlastYear", "OrderCount",
    "DaySinceLastOrder", "CashbackAmount"
]

for col in numeric_eda:
    plt.figure(figsize=(7, 4))
    plt.hist(df[col].dropna(), bins=30)
    plt.title(f"Distribuição — {col}")
    plt.xlabel(col)
    plt.ylabel("Frequência")
    plt.tight_layout()
    plt.show()

### EDA — Correlação de Pearson

A matriz de correlação considera as variáveis numéricas e auxilia na identificação de relações lineares e possíveis redundâncias.

In [ ]:
corr = df.select_dtypes(include=np.number).corr(method="pearson")

plt.figure(figsize=(11, 8))
plt.imshow(corr, aspect="auto")
plt.colorbar(label="Correlação de Pearson")
plt.xticks(range(len(corr.columns)), corr.columns, rotation=90)
plt.yticks(range(len(corr.index)), corr.index)
plt.title("Mapa de calor — Correlação de Pearson")
plt.tight_layout()
plt.show()

### EDA — Interpretação e tomada de decisão

A célula seguinte transforma os resultados observados em uma síntese textual.

In [ ]:
churn_rate = df["Churn"].mean() * 100
missing_cols = df.columns[df.isna().any()].tolist()

print(
    f"A base possui {df.shape[0]:,} registros e {df.shape[1]} variáveis. "
    f"A taxa de churn observada é de {churn_rate:.2f}%, indicando uma classe minoritária "
    f"que deve ser considerada durante a modelagem. Foram encontradas {len(missing_cols)} "
    f"colunas com valores ausentes. A preparação deverá tratar esses nulos sem utilizar "
    f"informações do conjunto de teste. A presença de valores extremos será avaliada por "
    f"IQR, com atenção especial ao KNN, que utiliza distâncias, enquanto a Árvore de "
    f"Decisão é menos sensível à escala das variáveis."
)

# Fase 2 — Tratamento e Limpeza (Data Prep)

Primeiro serão identificadas e removidas duplicidades. A imputação será feita de forma segura: os parâmetros serão ajustados exclusivamente no treino e depois aplicados ao teste.

In [ ]:
duplicates_before = df.duplicated().sum()
df = df.drop_duplicates().copy()

print(f"Duplicidades encontradas: {duplicates_before}")
print(f"Dimensões após remoção de duplicadas: {df.shape}")

missing_after = df.isna().sum()
display(missing_after[missing_after > 0].sort_values(ascending=False).to_frame("nulos"))

### Data Prep — Divisão estratificada antes do ajuste dos transformadores

O alvo é separado das preditoras e `CustomerID` é removido por ser um identificador, não uma característica comportamental. O `stratify=y` preserva a proporção de classes.

In [ ]:
X = df.drop(columns=["Churn", "CustomerID"])
y = df["Churn"].copy()

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y
)

print("Treino:", X_train_raw.shape)
print("Teste :", X_test_raw.shape)

print("\nDistribuição de Churn no treino:")
display(y_train.value_counts(normalize=True).sort_index().rename("proporcao"))

print("\nDistribuição de Churn no teste:")
display(y_test.value_counts(normalize=True).sort_index().rename("proporcao"))

### Data Prep — Imputação sem vazamento

Para variáveis numéricas será usada **mediana**, pois ela é menos sensível a outliers. Para variáveis categóricas será utilizada a categoria mais frequente. Os imputadores são ajustados somente com o treino.

In [ ]:
numeric_cols_raw = X_train_raw.select_dtypes(include=np.number).columns.tolist()
categorical_cols_raw = X_train_raw.select_dtypes(exclude=np.number).columns.tolist()

num_imputer = SimpleImputer(strategy="median")
cat_imputer = SimpleImputer(strategy="most_frequent")

X_train_num = pd.DataFrame(
    num_imputer.fit_transform(X_train_raw[numeric_cols_raw]),
    columns=numeric_cols_raw,
    index=X_train_raw.index
)

X_test_num = pd.DataFrame(
    num_imputer.transform(X_test_raw[numeric_cols_raw]),
    columns=numeric_cols_raw,
    index=X_test_raw.index
)

X_train_cat = pd.DataFrame(
    cat_imputer.fit_transform(X_train_raw[categorical_cols_raw]),
    columns=categorical_cols_raw,
    index=X_train_raw.index
)

X_test_cat = pd.DataFrame(
    cat_imputer.transform(X_test_raw[categorical_cols_raw]),
    columns=categorical_cols_raw,
    index=X_test_raw.index
)

X_train_prep = pd.concat([X_train_num, X_train_cat], axis=1)[X_train_raw.columns]
X_test_prep = pd.concat([X_test_num, X_test_cat], axis=1)[X_test_raw.columns]

print("Nulos após imputação — treino:", X_train_prep.isna().sum().sum())
print("Nulos após imputação — teste :", X_test_prep.isna().sum().sum())

### Data Prep — Identificação dos outliers pelo método IQR

O IQR será usado para quantificar valores abaixo de Q1−1,5×IQR ou acima de Q3+1,5×IQR. Isso não significa que todo ponto extremo seja erro; a decisão será feita considerando o impacto no modelo.

In [ ]:
outlier_rows = []

for col in numeric_cols_raw:
    s = X_train_prep[col]
    q1 = s.quantile(0.25)
    q3 = s.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    count = ((s < lower) | (s > upper)).sum()

    outlier_rows.append({
        "variavel": col,
        "Q1": q1,
        "Q3": q3,
        "IQR": iqr,
        "limite_inferior": lower,
        "limite_superior": upper,
        "qtd_outliers": int(count),
        "percentual": 100 * count / len(s)
    })

outlier_summary = (
    pd.DataFrame(outlier_rows)
    .sort_values("qtd_outliers", ascending=False)
)

display(outlier_summary)

### Data Prep — Boxplots e decisão de tratamento

Os boxplots mostram a presença de valores extremos. Em vez de excluir clientes — o que poderia remover comportamentos legítimos — será utilizado **clipping pelos limites de IQR calculados no treino**. Essa decisão reduz o impacto dos extremos nas distâncias do KNN sem descartar observações.

In [ ]:
for col in numeric_cols_raw:
    plt.figure(figsize=(8, 2.8))
    plt.boxplot(X_train_prep[col], vert=False)
    plt.title(f"Boxplot — {col}")
    plt.xlabel(col)
    plt.tight_layout()
    plt.show()

# Fase 3 — Feature Engineering

A variável obrigatória `cashback_por_pedido` será criada pela razão entre `CashbackAmount` e `OrderCount`. Como os nulos das variáveis originais já foram tratados, o cálculo ocorre sem propagação de `NaN`. Também será protegida a divisão por zero.

In [ ]:
for data in [X_train_prep, X_test_prep]:
    data["cashback_por_pedido"] = (
        data["CashbackAmount"] /
        data["OrderCount"].replace(0, np.nan)
    )

X_train_prep["cashback_por_pedido"] = (
    X_train_prep["cashback_por_pedido"]
    .replace([np.inf, -np.inf], np.nan)
)

X_test_prep["cashback_por_pedido"] = (
    X_test_prep["cashback_por_pedido"]
    .replace([np.inf, -np.inf], np.nan)
)

feature_median = X_train_prep["cashback_por_pedido"].median()

X_train_prep["cashback_por_pedido"] = (
    X_train_prep["cashback_por_pedido"].fillna(feature_median)
)

X_test_prep["cashback_por_pedido"] = (
    X_test_prep["cashback_por_pedido"].fillna(feature_median)
)

print("Nova variável criada: cashback_por_pedido")
display(
    X_train_prep[
        ["CashbackAmount", "OrderCount", "cashback_por_pedido"]
    ].head()
)

### Feature Engineering — Clipping de outliers com parâmetros do treino

Os limites são calculados somente no treino e reutilizados no teste. A nova variável também participa do tratamento, pois será utilizada pelo modelo.

In [ ]:
numeric_model_cols = X_train_prep.select_dtypes(include=np.number).columns.tolist()
outlier_bounds = {}

for col in numeric_model_cols:
    q1 = X_train_prep[col].quantile(0.25)
    q3 = X_train_prep[col].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr

    outlier_bounds[col] = (lower, upper)

    X_train_prep[col] = X_train_prep[col].clip(lower=lower, upper=upper)
    X_test_prep[col] = X_test_prep[col].clip(lower=lower, upper=upper)

print("Clipping concluído com limites derivados exclusivamente do treino.")

# Fase 4 — Encoding, Balanceamento e Escalonamento Seguro

As categorias serão convertidas por One-Hot Encoding. O balanceamento será feito por SMOTE exclusivamente no treino. Serão preparados dois conjuntos: um com StandardScaler para KNN e outro sem escalonamento para Árvore de Decisão.

In [ ]:
categorical_model_cols = (
    X_train_prep.select_dtypes(exclude=np.number).columns.tolist()
)

# Primeiro fazemos apenas o encoding, sem escalonar.
# O StandardScaler será ajustado somente depois do SMOTE,
# conforme exigido pelo enunciado.
preprocessor_model = ColumnTransformer(
    transformers=[
        ("num", "passthrough", numeric_model_cols),
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore", sparse_output=False),
            categorical_model_cols
        )
    ],
    remainder="drop"
)

X_train_encoded = preprocessor_model.fit_transform(X_train_prep)
X_test_encoded = preprocessor_model.transform(X_test_prep)

N_NUMERIC_FEATURES = len(numeric_model_cols)

print("Dimensão após encoding:", X_train_encoded.shape)
print("Variáveis numéricas contínuas:", N_NUMERIC_FEATURES)


### Balanceamento — SMOTE somente no conjunto de treino

O SMOTE sintetiza exemplos da classe minoritária. O teste permanece com a distribuição original para representar o cenário real de avaliação e evitar data leakage.

In [ ]:
smote = SMOTE(random_state=RANDOM_STATE)

# Balanceamento ocorre estritamente no treino.
X_train_bal_encoded, y_train_bal = smote.fit_resample(
    X_train_encoded, y_train
)

X_train_tree_bal = X_train_bal_encoded.copy()
y_train_tree_bal = y_train_bal.copy()
X_test_tree = X_test_encoded.copy()

# KNN: StandardScaler é ajustado SOMENTE no treino já balanceado.
scaler_knn = StandardScaler()

X_train_knn_bal = X_train_bal_encoded.copy()
X_test_knn = X_test_encoded.copy()

X_train_knn_bal[:, :N_NUMERIC_FEATURES] = scaler_knn.fit_transform(
    X_train_bal_encoded[:, :N_NUMERIC_FEATURES]
)

X_test_knn[:, :N_NUMERIC_FEATURES] = scaler_knn.transform(
    X_test_encoded[:, :N_NUMERIC_FEATURES]
)

print("Distribuição original do treino:")
print(y_train.value_counts().sort_index().to_dict())

print("\nDistribuição após SMOTE:")
print(pd.Series(y_train_bal).value_counts().sort_index().to_dict())

print("\nDistribuição mantida no teste:")
print(y_test.value_counts().sort_index().to_dict())

print(
    "\nStandardScaler: fit_transform aplicado no treino balanceado "
    "e transform aplicado no teste."
)


### Escalonamento seguro — justificativa

O KNN depende de distâncias, portanto as variáveis numéricas precisam estar em escalas comparáveis. O StandardScaler foi ajustado somente no treino. A Árvore não recebe scaler porque seus cortes são baseados em limiares e não dependem da distância euclidiana.

In [ ]:
print(
    "KNN: o StandardScaler é ajustado exclusivamente no treino "
    "já balanceado e depois aplicado ao teste."
)
print(
    "Árvore de Decisão: não recebe StandardScaler, pois seus cortes "
    "por limiar não dependem da escala das variáveis."
)


# Fase 5 — Modelagem e Validação: KNN

Serão testados quatro valores de `n_neighbors` — 3, 5, 7 e 9. Em cada cenário serão registradas métricas de treino e teste para observar generalização e possíveis sinais de overfitting.

In [ ]:
knn_results = []
knn_models = {}

for k in [3, 5, 7, 9]:
    model = KNeighborsClassifier(n_neighbors=k)
    model.fit(X_train_knn_bal, y_train_bal)

    pred_train = model.predict(X_train_knn_bal)
    pred_test = model.predict(X_test_knn)

    knn_results.append({
        "modelo": "KNN",
        "parametro": f"n_neighbors={k}",
        "treino_accuracy": accuracy_score(y_train_bal, pred_train),
        "teste_accuracy": accuracy_score(y_test, pred_test),
        "treino_precision": precision_score(y_train_bal, pred_train),
        "teste_precision": precision_score(y_test, pred_test),
        "treino_recall": recall_score(y_train_bal, pred_train),
        "teste_recall": recall_score(y_test, pred_test),
        "treino_f1": f1_score(y_train_bal, pred_train),
        "teste_f1": f1_score(y_test, pred_test),
        "gap_f1": (
            f1_score(y_train_bal, pred_train)
            - f1_score(y_test, pred_test)
        )
    })

    knn_models[k] = model

knn_results_df = pd.DataFrame(knn_results)
display(knn_results_df)

### Diagnóstico de overfitting — KNN

O gráfico compara F1 de treino e teste. Uma diferença grande entre as curvas indica perda de generalização.

In [ ]:
plt.figure(figsize=(9, 5))
plt.plot(
    [3, 5, 7, 9],
    knn_results_df["treino_f1"],
    marker="o",
    label="Treino F1"
)
plt.plot(
    [3, 5, 7, 9],
    knn_results_df["teste_f1"],
    marker="o",
    label="Teste F1"
)
plt.xlabel("n_neighbors (K)")
plt.ylabel("F1-score")
plt.title("KNN — F1 de Treino vs Teste")
plt.legend()
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()

best_knn_k = int(
    knn_results_df
    .sort_values(["teste_f1", "teste_recall"], ascending=False)
    .iloc[0]["parametro"]
    .split("=")[1]
)

print(f"Melhor configuração KNN pelo F1 de teste: K={best_knn_k}")

# Fase 5 — Modelagem e Validação: Árvore de Decisão

Serão testadas quatro profundidades: 3, 5, 7 e ilimitada (`None`). A comparação entre treino e teste permitirá observar o crescimento da complexidade e seu efeito sobre a generalização.

In [ ]:
tree_results = []
tree_models = {}

for depth in [3, 5, 7, None]:
    model = DecisionTreeClassifier(
        max_depth=depth,
        random_state=RANDOM_STATE
    )

    model.fit(X_train_tree_bal, y_train_tree_bal)

    pred_train = model.predict(X_train_tree_bal)
    pred_test = model.predict(X_test_tree)

    tree_results.append({
        "modelo": "Árvore de Decisão",
        "parametro": f"max_depth={depth}",
        "treino_accuracy": accuracy_score(y_train_tree_bal, pred_train),
        "teste_accuracy": accuracy_score(y_test, pred_test),
        "treino_precision": precision_score(y_train_tree_bal, pred_train),
        "teste_precision": precision_score(y_test, pred_test),
        "treino_recall": recall_score(y_train_tree_bal, pred_train),
        "teste_recall": recall_score(y_test, pred_test),
        "treino_f1": f1_score(y_train_tree_bal, pred_train),
        "teste_f1": f1_score(y_test, pred_test),
        "gap_f1": (
            f1_score(y_train_tree_bal, pred_train)
            - f1_score(y_test, pred_test)
        )
    })

    tree_models[depth] = model

tree_results_df = pd.DataFrame(tree_results)
display(tree_results_df)

### Diagnóstico de overfitting — Árvore de Decisão

A profundidade ilimitada tende a aumentar a capacidade de memorizar o treino. O gráfico permite avaliar se esse aumento de complexidade trouxe ganho de generalização.

In [ ]:
depth_labels = ["3", "5", "7", "None"]

plt.figure(figsize=(9, 5))
plt.plot(
    depth_labels,
    tree_results_df["treino_f1"],
    marker="o",
    label="Treino F1"
)
plt.plot(
    depth_labels,
    tree_results_df["teste_f1"],
    marker="o",
    label="Teste F1"
)
plt.xlabel("max_depth")
plt.ylabel("F1-score")
plt.title("Árvore de Decisão — F1 de Treino vs Teste")
plt.legend()
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()

best_tree_depth = (
    tree_results_df
    .sort_values(["teste_f1", "teste_recall"], ascending=False)
    .iloc[0]["parametro"]
    .split("=")[1]
)

best_tree_depth_value = (
    None if best_tree_depth == "None"
    else int(best_tree_depth)
)

print(
    f"Melhor configuração da Árvore pelo F1 de teste: "
    f"max_depth={best_tree_depth_value}"
)

### Comparação das duas melhores configurações

Nesta etapa são selecionados o melhor KNN e a melhor Árvore pelo F1 de teste. O F1 equilibra precisão e recall e evita que a seleção dependa exclusivamente de acurácia em uma base com classes desbalanceadas.

In [ ]:
best_knn_row = knn_results_df[
    knn_results_df["parametro"] == f"n_neighbors={best_knn_k}"
].iloc[0]

best_tree_row = tree_results_df[
    tree_results_df["parametro"] == f"max_depth={best_tree_depth_value}"
].iloc[0]

comparison = pd.DataFrame([
    {
        "modelo": f"KNN (K={best_knn_k})",
        "teste_accuracy": best_knn_row["teste_accuracy"],
        "teste_precision": best_knn_row["teste_precision"],
        "teste_recall": best_knn_row["teste_recall"],
        "teste_f1": best_knn_row["teste_f1"],
        "gap_f1": best_knn_row["gap_f1"]
    },
    {
        "modelo": f"Árvore (depth={best_tree_depth_value})",
        "teste_accuracy": best_tree_row["teste_accuracy"],
        "teste_precision": best_tree_row["teste_precision"],
        "teste_recall": best_tree_row["teste_recall"],
        "teste_f1": best_tree_row["teste_f1"],
        "gap_f1": best_tree_row["gap_f1"]
    }
])

display(comparison)

# Fase 6 — Avaliação e Veredito de Negócios

Agora serão gerados o `Classification Report` e as matrizes de confusão das duas configurações selecionadas. Para churn, o **Falso Negativo (FN)** representa um cliente que abandonaria a plataforma, mas não seria identificado pela campanha preventiva. O **Falso Positivo (FP)** representa um cliente que receberia uma ação de retenção embora não estivesse destinado a abandonar.

In [ ]:
best_knn_model = knn_models[best_knn_k]
best_tree_model = tree_models[best_tree_depth_value]

pred_knn = best_knn_model.predict(X_test_knn)
pred_tree = best_tree_model.predict(X_test_tree)

print(f"CLASSIFICATION REPORT — KNN (K={best_knn_k})")
print(
    classification_report(
        y_test,
        pred_knn,
        target_names=["Não Churn", "Churn"]
    )
)

print(
    f"CLASSIFICATION REPORT — "
    f"Árvore (max_depth={best_tree_depth_value})"
)
print(
    classification_report(
        y_test,
        pred_tree,
        target_names=["Não Churn", "Churn"]
    )
)

### Matrizes de confusão

As matrizes mostram explicitamente TN, FP, FN e TP. Essa leitura é essencial para conectar o desempenho estatístico ao custo operacional de uma campanha de retenção.

In [ ]:
cm_knn = confusion_matrix(y_test, pred_knn)
cm_tree = confusion_matrix(y_test, pred_tree)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, cm, title in [
    (axes[0], cm_knn, f"KNN (K={best_knn_k})"),
    (axes[1], cm_tree, f"Árvore (depth={best_tree_depth_value})")
]:
    ax.imshow(cm)
    ax.set_title(f"Matriz de Confusão — {title}")
    ax.set_xlabel("Predito")
    ax.set_ylabel("Real")
    ax.set_xticks([0, 1], ["Não Churn", "Churn"])
    ax.set_yticks([0, 1], ["Não Churn", "Churn"])

    for i in range(2):
        for j in range(2):
            ax.text(j, i, cm[i, j], ha="center", va="center")

plt.tight_layout()
plt.show()

tn_knn, fp_knn, fn_knn, tp_knn = cm_knn.ravel()
tn_tree, fp_tree, fn_tree, tp_tree = cm_tree.ravel()

print(
    "KNN  -> TN:", tn_knn,
    "FP:", fp_knn,
    "FN:", fn_knn,
    "TP:", tp_knn
)

print(
    "Árvore -> TN:", tn_tree,
    "FP:", fp_tree,
    "FN:", fn_tree,
    "TP:", tp_tree
)

### Veredito de negócio

O projeto adota a premissa operacional de que, em uma estratégia de retenção, deixar um cliente realmente em churn sem nenhuma ação preventiva (FN) pode representar uma perda maior do que oferecer um cupom a um cliente que permaneceria (FP). A decisão final deve observar o equilíbrio entre recall, precisão, F1 e quantidade de FN/FP.

In [ ]:
metrics_business = pd.DataFrame([
    {
        "modelo": f"KNN (K={best_knn_k})",
        "recall": recall_score(y_test, pred_knn),
        "precision": precision_score(y_test, pred_knn),
        "f1": f1_score(y_test, pred_knn),
        "falsos_positivos": fp_knn,
        "falsos_negativos": fn_knn
    },
    {
        "modelo": f"Árvore (depth={best_tree_depth_value})",
        "recall": recall_score(y_test, pred_tree),
        "precision": precision_score(y_test, pred_tree),
        "f1": f1_score(y_test, pred_tree),
        "falsos_positivos": fp_tree,
        "falsos_negativos": fn_tree
    }
])

display(metrics_business)

if (
    fn_knn < fn_tree
    and recall_score(y_test, pred_knn) >= recall_score(y_test, pred_tree)
):
    business_verdict = f"KNN (K={best_knn_k})"
    business_reason = (
        f"O KNN identificou mais clientes em churn "
        f"(recall={recall_score(y_test, pred_knn):.3f}) "
        f"e produziu menos falsos negativos ({fn_knn}) "
        f"do que a Árvore ({fn_tree}). Isso reduz o número "
        f"de clientes em risco que passariam sem uma ação preventiva, "
        f"embora aumente os falsos positivos ({fp_knn}) e, "
        f"consequentemente, o volume potencial de cupons."
    )
else:
    business_verdict = (
        f"Árvore de Decisão (max_depth={best_tree_depth_value})"
    )
    business_reason = (
        f"A Árvore apresentou o melhor equilíbrio operacional "
        f"sob as métricas observadas, com recall="
        f"{recall_score(y_test, pred_tree):.3f}, precisão="
        f"{precision_score(y_test, pred_tree):.3f}, F1="
        f"{f1_score(y_test, pred_tree):.3f}, "
        f"{fp_tree} falsos positivos e {fn_tree} falsos negativos."
    )

print("VEREDITO DE NEGÓCIO:", business_verdict)
print(business_reason)

## Conclusão técnica

A conclusão consolida as decisões metodológicas: EDA, tratamento de nulos, duplicidades, outliers, engenharia da variável calculada, split estratificado, SMOTE somente no treino, escalonamento exclusivo do KNN, otimização dos hiperparâmetros e análise de erros.

In [ ]:
print(f"""
RESUMO FINAL
------------
Base: {CSV_NAME}
Registros finais: {len(df)}
Taxa de churn: {df["Churn"].mean():.2%}

Melhor KNN: K={best_knn_k}
- Accuracy teste: {accuracy_score(y_test, pred_knn):.4f}
- Precision teste: {precision_score(y_test, pred_knn):.4f}
- Recall teste: {recall_score(y_test, pred_knn):.4f}
- F1 teste: {f1_score(y_test, pred_knn):.4f}

Melhor Árvore: max_depth={best_tree_depth_value}
- Accuracy teste: {accuracy_score(y_test, pred_tree):.4f}
- Precision teste: {precision_score(y_test, pred_tree):.4f}
- Recall teste: {recall_score(y_test, pred_tree):.4f}
- F1 teste: {f1_score(y_test, pred_tree):.4f}

Veredito de negócio: {business_verdict}
""")

# Geração do README.md + ilustração do projeto

A última etapa cria automaticamente uma ilustração da arquitetura do pipeline e um `README.md` pronto para ser colocado no repositório GitHub. O README incorpora os resultados efetivamente obtidos na execução do notebook.

In [ ]:
architecture_path = "pipeline_projeto.png"

plt.figure(figsize=(12, 4))
steps = [
    "CSV",
    "EDA",
    "Data Prep",
    "Feature\nEngineering",
    "Split\nEstratificado",
    "SMOTE\n(train)",
    "KNN / Árvore",
    "Avaliação",
    "Veredito"
]

x = np.arange(len(steps))
plt.plot(x, np.zeros(len(steps)), marker="o")

for i, step in enumerate(steps):
    plt.text(
        i, 0.08, step,
        ha="center",
        va="bottom",
        fontsize=10
    )

    if i < len(steps) - 1:
        plt.annotate(
            "",
            xy=(i + 0.85, 0),
            xytext=(i + 0.15, 0),
            arrowprops=dict(arrowstyle="->")
        )

plt.yticks([])
plt.xticks([])
plt.title("Pipeline do Projeto — E-commerce Churn")
plt.box(False)
plt.tight_layout()
plt.savefig(
    architecture_path,
    dpi=180,
    bbox_inches="tight"
)
plt.show()

dictionary_rows = []

for col in df.columns:
    role = (
        "Alvo" if col == "Churn"
        else ("Identificador" if col == "CustomerID" else "Preditor")
    )

    dictionary_rows.append({
        "variavel": col,
        "tipo": str(df[col].dtype),
        "papel": role,
        "nulos_antes_do_tratamento": int(
            df_raw[col].isna().sum()
        )
    })

dictionary_df = pd.DataFrame(dictionary_rows)

readme = f"""
# Projeto Machine Learning — E-commerce Churn

## 1. Descrição do projeto

Este projeto implementa um pipeline de Machine Learning para prever **churn de clientes em um aplicativo de e-commerce**. O objetivo de negócio é identificar clientes com maior probabilidade de abandonar a plataforma para possibilitar ações preventivas de retenção, como ofertas ou cupons.

A base utilizada é **`E Commerce Dataset - E Comm.csv`**.

## 2. Objetivo de negócio

O problema é uma classificação binária:

- `Churn = 0`: cliente não abandonou a plataforma.
- `Churn = 1`: cliente abandonou a plataforma.

Em uma campanha de retenção, dois erros são relevantes:

- **Falso Positivo (FP):** o cliente recebe uma ação de retenção mesmo não estando em churn.
- **Falso Negativo (FN):** o cliente estava em churn, mas o modelo não o identificou.

Neste projeto, o FN recebe atenção especial porque representa uma oportunidade de retenção perdida.

## 3. Arquitetura do projeto

![Pipeline do projeto](pipeline_projeto.png)

## 4. Fases implementadas

### Fase 1 — EDA

Foram analisados:

- dimensões da base;
- tipos de dados;
- valores ausentes;
- estatísticas descritivas;
- distribuição da variável `Churn`;
- distribuições de variáveis numéricas;
- correlação de Pearson;
- presença de valores discrepantes.

A base final possui **{len(df):,} registros e {df.shape[1]} colunas**, com taxa de churn de **{df["Churn"].mean():.2%}**.

### Fase 2 — Data Prep

- duplicidades foram verificadas;
- valores nulos numéricos foram tratados com **mediana**;
- valores nulos categóricos foram tratados com a categoria mais frequente;
- outliers foram identificados pelo IQR;
- os extremos foram tratados por clipping usando limites calculados somente no conjunto de treino;
- `CustomerID` foi removido das variáveis preditoras por ser um identificador.

### Fase 3 — Feature Engineering

Foi criada obrigatoriamente a variável:

`cashback_por_pedido = CashbackAmount / OrderCount`

O cálculo foi protegido contra divisão por zero e valores infinitos.

### Fase 4 — Separação, Balanceamento e Escalonamento

O dataset foi dividido em:

- **80% treino**
- **20% teste**
- `stratify=y`

O SMOTE foi aplicado **somente ao treino**, evitando data leakage.

Para o KNN, foi aplicado `StandardScaler` às variáveis numéricas. Para a Árvore de Decisão, o escalonamento foi dispensado.

### Fase 5 — Modelagem

#### KNN

Foram testados:

- K = 3
- K = 5
- K = 7
- K = 9

Melhor configuração pelo F1 de teste: **K = {best_knn_k}**.

#### Árvore de Decisão

Foram testados:

- `max_depth = 3`
- `max_depth = 5`
- `max_depth = 7`
- `max_depth = None`

Melhor configuração pelo F1 de teste: **max_depth = {best_tree_depth_value}**.

## 5. Resultados

### KNN — melhor configuração

| Métrica | Resultado |
|---|---:|
| Accuracy | {accuracy_score(y_test, pred_knn):.4f} |
| Precision | {precision_score(y_test, pred_knn):.4f} |
| Recall | {recall_score(y_test, pred_knn):.4f} |
| F1-score | {f1_score(y_test, pred_knn):.4f} |
| Falsos positivos | {fp_knn} |
| Falsos negativos | {fn_knn} |

### Árvore — melhor configuração

| Métrica | Resultado |
|---|---:|
| Accuracy | {accuracy_score(y_test, pred_tree):.4f} |
| Precision | {precision_score(y_test, pred_tree):.4f} |
| Recall | {recall_score(y_test, pred_tree):.4f} |
| F1-score | {f1_score(y_test, pred_tree):.4f} |
| Falsos positivos | {fp_tree} |
| Falsos negativos | {fn_tree} |

## 6. Diagnóstico de overfitting

O diagnóstico foi realizado comparando métricas de treino e teste para diferentes níveis de complexidade.

A análise considera F1-score, recall, precisão e o gap entre treino e teste para avaliar a capacidade de generalização.

## 7. Veredito de negócio

**Modelo indicado no cenário de retenção: {business_verdict}**

{business_reason}

O resultado deve ser interpretado dentro do contexto operacional da empresa: reduzir FN pode aumentar FP, elevando o número de clientes que recebem incentivos sem necessidade. Portanto, a política de custo real de cupons e valor esperado de retenção deve ser monitorada após a implantação.

## 8. Dicionário de dados

| Variável | Tipo | Papel | Nulos na base original |
|---|---|---|---:|
"""

for _, row in dictionary_df.iterrows():
    readme += (
        f'| `{row["variavel"]}` | `{row["tipo"]}` | '
        f'{row["papel"]} | {row["nulos_antes_do_tratamento"]} |\n'
    )

readme += """
**Variável criada no projeto:** `cashback_por_pedido` — razão entre `CashbackAmount` e `OrderCount`.

## 9. Como baixar e executar

### Google Colab

1. Abra o arquivo `.ipynb` no Google Colab.
2. Execute as células em ordem.
3. Na célula de carregamento, envie `E Commerce Dataset - E Comm.csv`.
4. Aguarde a execução das seis fases.
5. Ao final, a última célula gerará:
   - `README.md`
   - `pipeline_projeto.png`
6. Baixe esses arquivos pelo painel de arquivos do Colab e coloque-os no repositório GitHub.

### Execução local

O notebook foi desenvolvido para Google Colab. Para execução local, instale as dependências:

```bash
pip install pandas numpy matplotlib scikit-learn imbalanced-learn jupyter
```

Depois abra o notebook com Jupyter ou VS Code.

## 10. Estrutura sugerida do GitHub

```text
projeto-machine-learning/
├── E Commerce Dataset - E Comm.csv
├── Projeto_Machine_Learning_Ecommerce_Churn.ipynb
├── README.md
└── pipeline_projeto.png
```

## 11. Boas práticas de versionamento

O projeto deve ser desenvolvido em branches por fase, por exemplo:

- `fase/eda`
- `fase/data-prep`
- `fase/feature-engineering`
- `fase/balanceamento`
- `fase/modelagem`
- `fase/avaliacao`

Exemplos de commits semânticos:

```text
feat: adiciona análise exploratória
feat: implementa imputação de valores nulos
feat: cria cashback_por_pedido
feat: adiciona balanceamento com SMOTE
feat: implementa testes de KNN
feat: implementa testes da árvore de decisão
feat: adiciona matrizes de confusão
docs: adiciona README do projeto
```

## 12. Conclusão

O projeto demonstra um fluxo completo de Ciência de Dados aplicado a um problema de negócio real: preparação dos dados, engenharia de atributos, prevenção de data leakage, balanceamento, escalonamento específico por algoritmo, comparação de hiperparâmetros, diagnóstico de overfitting e análise dos erros.

A conclusão final não deve ser baseada apenas em acurácia. Para churn, é fundamental compreender o impacto de falsos positivos e falsos negativos e alinhar a escolha do modelo à estratégia financeira e operacional da empresa.
"""

with open("README.md", "w", encoding="utf-8") as f:
    f.write(readme)

print("Arquivos gerados:")
print("-", os.path.abspath("README.md"))
print("-", os.path.abspath(architecture_path))

## Checklist de entrega

Antes de enviar o projeto, confirme:

- [ ] Notebook executa do início ao fim no Google Colab.
- [ ] CSV correto foi utilizado.
- [ ] Todas as 6 fases estão claramente identificadas.
- [ ] Cada célula de código possui uma célula Markdown anterior explicando sua finalidade.
- [ ] EDA contém pelo menos 3 visualizações explicativas.
- [ ] Duplicidades, nulos e outliers foram tratados e justificados.
- [ ] `cashback_por_pedido` foi criada.
- [ ] Split utiliza `test_size=0.20` e `stratify=y`.
- [ ] SMOTE foi aplicado somente no treino.
- [ ] StandardScaler foi utilizado no KNN e não na Árvore.
- [ ] Foram testados pelo menos 4 valores de K e 4 valores de `max_depth`.
- [ ] Treino e teste foram comparados para diagnosticar overfitting.
- [ ] Classification Report e matrizes de confusão foram gerados.
- [ ] O veredito foi conectado aos FP/FN e ao contexto de negócio.
- [ ] A última célula gerou `README.md` e a ilustração.
- [ ] O GitHub foi organizado com branches e commits semânticos.
- [ ] O vídeo de apresentação está dentro do limite de 7 minutos.